# Baseline: прогноз оценки отзыва

Сравним постоянный прогноз с TF-IDF и логистической регрессией, проверим дополнительные признаки и сохраним выбранную модель для API.

Используем только размеченный `train.csv`. Все словари и масштабирование обучаются внутри соответствующей обучающей части.

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from threadpoolctl import threadpool_limits

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

sys.path.insert(0, str(root))

In [2]:
from company_reviews.baseline import (
    median_from_probabilities,
    normalize_reviews_for_split,
    numeric_feature_matrix,
)
from company_reviews.training import (
    build_pipeline,
    negative_median_mae,
    save_models,
)

# Ограничиваем число потоков для компьютера с 8 ГБ памяти.
thread_limits = threadpool_limits(limits=2)
train = pd.read_csv(root / "data/raw/train.csv")

assert train["Review"].notna().all()
assert train["Rating"].isin(range(1, 6)).all()

## 1. Повторы и разделение данных

Сначала удаляем строки с **точно таким же `Review`**: иначе повторный отзыв получил бы дополнительный вес при обучении. У этих повторов оценки совпадают.

Чтобы не разделить отзывы, совпадающие после нормализации, между обучением и проверкой, создаём служебную копию каждого оставшегося текста: приводим буквы к одному регистру и схлопываем пробелы. Если две такие копии равны, присваиваем отзывам **один номер группы**. Это не группировка по теме, смыслу или оценке. Номер нужен только при разделении данных; модель получает исходный текст `Review`.

Ниже показаны число групп и единственная пара отзывов, которая совпала после такой нормализации.

In [3]:
duplicate_labels = (
    train.loc[train.duplicated("Review", keep=False)]
    .groupby("Review")["Rating"]
    .nunique()
)
assert (duplicate_labels <= 1).all()

train_clean = train.drop_duplicates(subset="Review", keep="first").reset_index(
    drop=True
)
normalized_reviews = normalize_reviews_for_split(train_clean["Review"])
group_numbers = pd.Series(
    pd.factorize(normalized_reviews, sort=True)[0] + 1,
    index=train_clean.index,
    name="номер группы",
)
group_sizes = group_numbers.value_counts()

print(f"Удалено строк с повторным Review: {len(train) - len(train_clean)}")
print(f"Осталось отзывов: {len(train_clean):,}")
print(f"Групп: {len(group_sizes):,}")
print(f"Групп из одного отзыва: {(group_sizes == 1).sum():,}")
print(f"Групп из двух и более отзывов: {(group_sizes > 1).sum():,}")

paired_rows = group_numbers[group_numbers.duplicated(keep=False)].index
example = train_clean.loc[paired_rows, ["Review", "Rating"]].copy()
example["после нормализации"] = normalized_reviews.loc[paired_rows]
example["номер группы"] = group_numbers.loc[paired_rows]
display(example.reset_index(drop=True))

Удалено строк с повторным Review: 24
Осталось отзывов: 59,976
Групп: 59,975
Групп из одного отзыва: 59,974
Групп из двух и более отзывов: 1


,Review,Rating,после нормализации,номер группы
0,"Great product, delivered on time.",5,"great product, delivered on time.",20124
1,"great product, delivered on time.",5,"great product, delivered on time.",20124


Обе строки в таблице получили один номер группы. Сами номера произвольны: важно только, совпадают они или нет. Здесь групп из нескольких отзывов всего одна. Такое разделение не даст двум вариантам этого отзыва оказаться по разные стороны проверки.

`StratifiedGroupKFold` по возможности сохраняет доли оценок 1–5 и держит отзывы одной группы вместе. Фолд — одна из пяти проверочных частей. Для коротких опытов берём первый фолд и затем ещё раз делим его обучающую часть. В итоговом сравнении каждый из пяти фолдов по очереди станет проверочным.

In [4]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2026)
train_idx, valid_idx = next(
    splitter.split(train_clean["Review"], train_clean["Rating"], group_numbers)
)

train_part = train_clean.iloc[train_idx]
valid_part = train_clean.iloc[valid_idx]
assert set(group_numbers.iloc[train_idx]).isdisjoint(set(group_numbers.iloc[valid_idx]))

print(f"Обучение: {len(train_part):,}; проверка: {len(valid_part):,}")
class_share = pd.DataFrame(
    {
        "обучение, %": 100
        * train_part["Rating"].value_counts(normalize=True).sort_index(),
        "проверка, %": 100
        * valid_part["Rating"].value_counts(normalize=True).sort_index(),
    }
)
display(class_share.round(2).rename_axis("Rating"))

Обучение: 47,980; проверка: 11,996


,"обучение, %","проверка, %"
Rating,,
1,31.11,31.11
2,2.72,2.72
3,2.80,2.80
4,5.58,5.59
5,57.79,57.79


## 2. Постоянный прогноз: почему медиана

Самая простая модель не учитывает текст и выдаёт одну оценку для всех отзывов. Выберем её так, чтобы MAE на обучающих данных была минимальной.

Для этого подходит **медиана — середина отсортированного списка оценок**.

Если сдвинуть прогноз от медианы на один балл в любую сторону, ошибка увеличится как минимум у половины отзывов, а уменьшится — не более чем у половины. Поэтому суммарная ошибка не станет меньше. Для MAE вывод тот же: деление на число отзывов ничего не меняет.

**На каждом фолде вычисляем медиану только по обучающим оценкам и прогнозируем её для всех отзывов проверочной части.** Так получаем ориентир, с которым сравним текстовую модель.

In [5]:
y_train = train_part["Rating"].to_numpy()
y_valid = valid_part["Rating"].to_numpy()

dummy = DummyRegressor(strategy="median")
dummy.fit(np.zeros((len(train_part), 1)), y_train)
dummy_pred = dummy.predict(np.zeros((len(valid_part), 1)))
dummy_mae = mean_absolute_error(y_valid, dummy_pred)

print(f"Постоянный прогноз: {dummy_pred[0]:.0f}")
print(f"MAE на проверке: {dummy_mae:.5f}")

Постоянный прогноз: 5
MAE на проверке: 1.43781


## 3. Слова и биграммы

TF-IDF создаёт столбцы для отдельных слов и пар соседних слов: `good`, `service`, `good service`. Одна строка матрицы соответствует отзыву.

- `ngram_range=(1, 2)` — слова и биграммы.
- `min_df=3` — начальный порог: признак должен встретиться хотя бы в трёх обучающих отзывах.
- `max_features=150_000` — общий предел числа разных слов и биграмм; ограничение памяти, а не подобранный оптимум.
- `sublinear_tf=True` — многократный повтор слова увеличивает его вес медленнее.
- `C=4` — начальная настройка силы регуляризации; ниже проверим соседние значения.

Логистическая регрессия оценивает вероятности пяти оценок. Для каждого отзыва выбираем первую оценку, на которой накопленная вероятность достигает 0,5. Это медиана предсказанного распределения: доказательство выше сохраняется, если заменить числа наблюдений их вероятностями. Такой ответ минимизирует ожидаемую абсолютную ошибку при полученных вероятностях.

In [6]:
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=150_000,
    sublinear_tf=True,
    dtype=np.float32,
)
word_classifier = LogisticRegression(C=4, max_iter=250, solver="lbfgs")
word_model = Pipeline(
    [
        ("tfidf", word_vectorizer),
        ("classifier", word_classifier),
    ]
)
word_model.fit(train_part["Review"], y_train)

probabilities = word_model.predict_proba(valid_part["Review"])
classes = word_model.named_steps["classifier"].classes_
word_pred = median_from_probabilities(probabilities, classes)
word_mae = mean_absolute_error(y_valid, word_pred)

display(
    pd.DataFrame(
        {
            "модель": ["Постоянная медиана", "Слова TF-IDF + логистическая регрессия"],
            "MAE": [dummy_mae, word_mae],
        }
    ).round(5)
)

,модель,MAE
0,Постоянная медиана,1.43781
1,Слова TF-IDF + логистическая регрессия,0.19165


На первом разбиении текстовая модель лучше постоянного прогноза. Отдельно проверим оценки 2–4: их мало, и общая MAE может скрывать ошибки на этих классах.

In [7]:
rating_summary = pd.DataFrame(
    {
        "отзывов": [(y_valid == rating).sum() for rating in classes],
        "MAE": [
            mean_absolute_error(
                y_valid[y_valid == rating], word_pred[y_valid == rating]
            )
            for rating in classes
        ],
    },
    index=pd.Index(classes, name="Rating"),
)
display(rating_summary.round(4))

,отзывов,MAE
Rating,,
1,3732,0.1294
2,326,1.0000
3,336,1.2976
4,670,0.8940
5,6932,0.0656


## 4. Короткие опыты с признаками

Сначала проверим варианты обработки текста и числовых признаков на одном разбиении — это быстрее, чем каждый раз обучать модель на пяти фолдах.

Для этих опытов берём обучающую часть первого фолда и дополнительно делим её на две части: на одной обучаем модели, на другой сравниваем их MAE.

Во всех опытах используем одни и те же отзывы для обучения и проверки. Меняем только исследуемую настройку или набор признаков.

В разделе 5 сравним выбранные варианты на всех пяти фолдах.

In [8]:
inner_part = train_part.reset_index(drop=True)
inner_group_numbers = group_numbers.iloc[train_idx].reset_index(drop=True)
inner_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2027)
fit_idx, check_idx = next(
    inner_splitter.split(
        inner_part["Review"], inner_part["Rating"], inner_group_numbers
    )
)

fit_part = inner_part.iloc[fit_idx]
check_part = inner_part.iloc[check_idx]
assert set(inner_group_numbers.iloc[fit_idx]).isdisjoint(
    set(inner_group_numbers.iloc[check_idx])
)
print(f"Внутреннее обучение: {len(fit_part):,}; проверка: {len(check_part):,}")

Внутреннее обучение: 38,384; проверка: 9,596


### 4.1. Нужно ли удалять стоп-слова

Проверим пять вариантов на одном внутреннем разбиении.

Функция ниже только переводит вероятности в оценки и считает MAE.

In [9]:
def validation_mae(model):
    probabilities = model.predict_proba(check_part["Review"])
    prediction = median_from_probabilities(probabilities, model.classes_)
    return mean_absolute_error(check_part["Rating"], prediction)

- **все слова**

In [10]:
all_words = clone(word_model)

all_words.fit(fit_part["Review"], fit_part["Rating"])
all_words_mae = validation_mae(all_words)
print(f"MAE: {all_words_mae:.5f}")

MAE: 0.19675


- **удаляем стандартный английский список**

In [11]:
without_stopwords = clone(word_model)
without_stopwords.set_params(tfidf__stop_words="english")

without_stopwords.fit(fit_part["Review"], fit_part["Rating"])
without_stopwords_mae = validation_mae(without_stopwords)
print(f"MAE: {without_stopwords_mae:.5f}")

MAE: 0.22614


- **сохраняем отрицания**

In [12]:
with_negations = clone(word_model)
with_negations.set_params(
    tfidf__stop_words=sorted(ENGLISH_STOP_WORDS - {"no", "not", "never", "nor"})
)

with_negations.fit(fit_part["Review"], fit_part["Rating"])
with_negations_mae = validation_mae(with_negations)
print(f"MAE: {with_negations_mae:.5f}")

MAE: 0.21728


- **сохраняем отрицания и слова контекста**

In [13]:
with_context = clone(word_model)
with_context.set_params(
    tfidf__stop_words=sorted(
        ENGLISH_STOP_WORDS
        - {"no", "not", "never", "nor", "but", "however", "very", "without"}  # вот слова контекста
    )
)

with_context.fit(fit_part["Review"], fit_part["Rating"])
with_context_mae = validation_mae(with_context)
print(f"MAE: {with_context_mae:.5f}")

MAE: 0.20873


**Исключаем из словаря слова и биграммы, встречающиеся более чем в 30% обучающих отзывов**

In [14]:
without_frequent = clone(word_model)
without_frequent.set_params(tfidf__max_df=0.3)

without_frequent.fit(fit_part["Review"], fit_part["Rating"])
without_frequent_mae = validation_mae(without_frequent)
print(f"MAE: {without_frequent_mae:.5f}")

MAE: 0.19644


**Сравнение обработки слов.**

| вариант | MAE |
| --- | --- |
| Все слова | 0.19675 |
| Удалены английские стоп-слова | 0.22614 |
| Стоп-слова удалены, отрицания сохранены | 0.21728 |
| Сохранены отрицания и связки | 0.20873 |
| Удалены термины из >30% отзывов | 0.19644 |

Список стоп-слов в дальнейшей модели не применяем: его удаление ухудшило MAE. Небольшое преимущество `max_df=0.3` на одном разбиении не считаем достаточным основанием менять базовый вариант.

### 4.2. Признаки из EDA и текста

Четыре признака из EDA: длина в символах, число слов, наличие `!` и наличие `?`. Для длины применяем `log1p`: так очень длинный отзыв меньше отличается по масштабу от обычного.

Дополнительно проверим число слов целиком заглавными буквами (`GREAT`), количество `!` и `?`, а также шесть индикаторов явно написанной оценки 0–5 (`one star`, `5/5`). Всего получается 13 числовых столбцов. Это измеримые свойства отзыва; настоящая оценка `Rating` в них не входит.

В `build_pipeline` объединены TF-IDF, масштабирование числовых столбцов и классификатор. Масштабирование использует только обучающие отзывы. После него числовые значения умножаются на 0,1; этот вес проверим отдельно.

In [15]:
feature_examples = pd.Series(["WOW!! one star?", "great company 5/5"])
example_features = numeric_feature_matrix(feature_examples)
display(
    pd.DataFrame(
        {
            "отзыв": feature_examples,
            "прописных слов": np.expm1(example_features[:, 4]).astype(int),
            "знаков !": np.expm1(example_features[:, 5]).astype(int),
            "знаков ?": np.expm1(example_features[:, 6]).astype(int),
            "найдена оценка": np.where(
                example_features[:, 7:13].any(axis=1),
                example_features[:, 7:13].argmax(axis=1),
                "нет",
            ),
        }
    )
)

mentioned_flags = numeric_feature_matrix(train_clean["Review"])[:, 7:13]
print(f"Отзывов с распознанной оценкой: {int(mentioned_flags.any(axis=1).sum()):,}")

,отзыв,прописных слов,знаков !,знаков ?,найдена оценка
0,WOW!! one star?,1,2,1,1
1,great company 5/5,0,0,0,5


Отзывов с распознанной оценкой: 1,792


- **слова и четыре признака EDA** (`log1p` длины отзыва в символах и словах, наличие `!` и `?`)

In [16]:
eda_model = build_pipeline(min_df=3, c=4)
eda_model.set_params(
    features__char_weight=0,
    features__numeric_columns=(0, 1, 2, 3),
)

eda_model.fit(fit_part["Review"], fit_part["Rating"])
print(f"MAE: {validation_mae(eda_model):.5f}")

MAE: 0.19633


- **добавляем к EDA число слов заглавными буквами**

In [17]:
uppercase_model = build_pipeline(min_df=3, c=4)
uppercase_model.set_params(
    features__char_weight=0,
    features__numeric_columns=(0, 1, 2, 3, 4),
)

uppercase_model.fit(fit_part["Review"], fit_part["Rating"])
print(f"MAE: {validation_mae(uppercase_model):.5f}")

MAE: 0.19633


- **добавляем к EDA количество знаков ! и ?**

In [18]:
punctuation_model = build_pipeline(min_df=3, c=4)
punctuation_model.set_params(
    features__char_weight=0,
    features__numeric_columns=(0, 1, 2, 3, 5, 6),
)

punctuation_model.fit(fit_part["Review"], fit_part["Rating"])
print(f"MAE: {validation_mae(punctuation_model):.5f}")

MAE: 0.19696


- **добавляем к EDA упоминание оценки**

In [19]:
rating_mention_model = build_pipeline(min_df=3, c=4)
rating_mention_model.set_params(
    features__char_weight=0,
    features__numeric_columns=(0, 1, 2, 3, 7, 8, 9, 10, 11, 12),
)

rating_mention_model.fit(fit_part["Review"], fit_part["Rating"])
print(f"MAE: {validation_mae(rating_mention_model):.5f}")

MAE: 0.19539


- **слова и все 13 числовых признаков**

In [20]:
all_numeric_model = build_pipeline(min_df=3, c=4)
all_numeric_model.set_params(
    features__char_weight=0,
    features__numeric_columns=None,
)

all_numeric_model.fit(fit_part["Review"], fit_part["Rating"])
print(f"MAE: {validation_mae(all_numeric_model):.5f}")

MAE: 0.19446


**Сравнение числовых признаков.** Во всех строках остаётся один и тот же TF-IDF по словам и биграммам.

| Добавленные числовые признаки | MAE |
| --- | --- |
| Без числовых признаков | 0,19675 |
| Четыре из EDA | 0,19633 |
| EDA + заглавные слова | 0,19633 |
| EDA + количество ! и ? | 0,19696 |
| EDA + упоминание оценки | 0,19539 |
| Все 13 | 0,19446 |

Для дальнейшего сравнения оставляем **полный набор**. Пользу дополнений перепроверим на пяти фолдах.

## 5. Какие признаки помогают на пяти фолдах

Сравним пять наборов при одинаковых `C=4`, словном `min_df=5`, весе числовых признаков 0,1 и разбиениях. Это рабочие настройки для сравнения признаков; окончательный подбор будет в разделе 6. Каждому набору соответствует отдельная ячейка.

Символьный TF-IDF (`char_wb`) добавляет фрагменты длиной 3–5 символов внутри слов, например `ser`, `serv`, `servi`. Порог частоты — 5 отзывов, предел словаря — 100 000 фрагментов, вес — 0,5. Это фиксированные настройки для проверки гипотезы; отдельного полного поиска по ним не было.

`cross_val_predict` обучает модель на четырёх частях и прогнозирует пятую. В результате у каждого отзыва появляется один прогноз модели, которая его не видела при обучении. Предобработка выполняется заново внутри каждого фолда.

In [21]:
reviews = train_clean["Review"]
labels = train_clean["Rating"].to_numpy()
folds = list(splitter.split(reviews, labels, group_numbers))

# Pipeline сам переиспользует одинаковые преобразования обучающих фолдов.
memory = joblib.Memory(root / "data/cache/tfidf-transforms", verbose=0)


def predict_on_folds(model):
    probabilities = cross_val_predict(
        model,
        reviews,
        labels,
        cv=folds,
        method="predict_proba",
        n_jobs=1,
    )
    return median_from_probabilities(probabilities, np.arange(1, 6))


def mean_fold_mae(predictions):
    errors = [
        mean_absolute_error(labels[valid], predictions[valid]) for _, valid in folds
    ]
    return float(np.mean(errors))

In [22]:
dummy_predictions = cross_val_predict(
    DummyRegressor(strategy="median"),
    np.zeros((len(reviews), 1)),
    labels,
    cv=folds,
    n_jobs=1,
)
print(f"Медиана, средняя MAE: {mean_fold_mae(dummy_predictions):.5f}")

Медиана, средняя MAE: 1.43781


- **только слова и биграммы**

In [23]:
words_model = build_pipeline(word_only=True, memory=memory)

words_predictions = predict_on_folds(words_model)
print(f"Средняя MAE: {mean_fold_mae(words_predictions):.5f}")

Средняя MAE: 0.19008


- **слова и четыре признака EDA**

In [24]:
words_eda_model = build_pipeline(memory=memory)
words_eda_model.set_params(
    features__char_weight=0, features__numeric_columns=(0, 1, 2, 3)
)

words_eda_predictions = predict_on_folds(words_eda_model)
print(f"Средняя MAE: {mean_fold_mae(words_eda_predictions):.5f}")

Средняя MAE: 0.18946


- **слова и все 13 числовых признаков**

In [25]:
words_numeric_model = build_pipeline(memory=memory)
words_numeric_model.set_params(features__char_weight=0)

words_numeric_predictions = predict_on_folds(words_numeric_model)
print(f"Средняя MAE: {mean_fold_mae(words_numeric_predictions):.5f}")

Средняя MAE: 0.18751


- **слова и символьные фрагменты**

In [26]:
words_char_model = build_pipeline(memory=memory)
words_char_model.set_params(features__numeric_weight=0)

words_char_predictions = predict_on_folds(words_char_model)
print(f"Средняя MAE: {mean_fold_mae(words_char_predictions):.5f}")

Средняя MAE: 0.18869


- **TF-IDF по словам и символам + 13 числовых признаков**

In [27]:
full_model = build_pipeline(memory=memory)

full_predictions = predict_on_folds(full_model)
print(f"Средняя MAE: {mean_fold_mae(full_predictions):.5f}")

Средняя MAE: 0.18539


**Сравнение на пяти фолдах при C=4, min_df=5 и весе числовых признаков 0,1.**

| Признаки | Средняя MAE |
| --- | --- |
| Постоянная медиана | 1,43781 |
| Слова и биграммы | 0,19008 |
| Слова + четыре признака EDA | 0,18946 |
| Слова + 13 числовых признаков | 0,18751 |
| Слова + символьные фрагменты | 0,18869 |
| TF-IDF по словам и символам + 13 числовых признаков + Logistic Regression | 0,18539 |

Полный набор дал небольшой выигрыш: около 2,5% относительно модели только со словами. Его оставляем для итогового подбора.

### Ошибки по оценкам

Отдельно посмотрим оценки 2–4: их существенно меньше, и среднее качество по всем отзывам может скрывать ошибки на них.

In [28]:
rating_errors = pd.DataFrame(
    {
        "отзывов": [(labels == rating).sum() for rating in range(1, 6)],
        "только слова": [
            mean_absolute_error(
                labels[labels == rating], words_predictions[labels == rating]
            )
            for rating in range(1, 6)
        ],
        "TF-IDF по словам и символам + 13 числовых признаков": [
            mean_absolute_error(
                labels[labels == rating], full_predictions[labels == rating]
            )
            for rating in range(1, 6)
        ],
    },
    index=pd.Index(range(1, 6), name="Rating"),
)
display(rating_errors.round(4))

,отзывов,только слова,TF-IDF по словам и символам + 13 числовых признаков
Rating,,,
1,18660,0.1247,0.1268
2,1629,1.0516,1.0135
3,1679,1.2734,1.2245
4,3349,0.8546,0.8325
5,34659,0.0681,0.0651


Оценки 2–4 остаются слабым местом. В прежних отдельных опытах мягкое увеличение веса редких классов ухудшило общую MAE; результаты и условия сохранены в отчёте о решениях.

## 6. Итоговый подбор C, min_df и веса числовых признаков

`GridSearchCV` совместно подбирает `C` 2 / 4 / 8, словный `min_df` 2 / 3 / 5 и вес числовых признаков 0,05 / 0,1 / 0,2: все 27 сочетаний на пяти фолдах, 135 обучений и финальное обучение лучшего варианта на всём train.

`negative_median_mae` считает MAE наших прогнозов и меняет знак. Это нужно потому, что `GridSearchCV` выбирает наибольшую оценку: −0,18 лучше −0,23. В таблицах показываем обычную положительную MAE.

Кэш `Pipeline` хранит преобразования, чтобы не строить один словарь заново для каждого `C`. Результат завершённого поиска автоматически не подгружается.

In [29]:
search = GridSearchCV(
    estimator=build_pipeline(memory=memory),
    param_grid={
        "classifier__C": [2, 4, 8],
        "features__min_df": [2, 3, 5],
        "features__numeric_weight": [0.05, 0.1, 0.2],
    },
    scoring=negative_median_mae,
    cv=folds,
    n_jobs=1,
    refit=True,
    error_score="raise",
    verbose=1,
)

search.fit(reviews, labels)

results = pd.DataFrame(search.cv_results_)
grid_table = pd.DataFrame(
    {
        "C": results["param_classifier__C"],
        "min_df": results["param_features__min_df"],
        "вес числовых признаков": results["param_features__numeric_weight"],
        "средняя MAE": -results["mean_test_score"],
        "разброс по фолдам": results["std_test_score"],
    }
)
display(grid_table.sort_values("средняя MAE").round(5))
print("Выбранные параметры:", search.best_params_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


,C,min_df,вес числовых признаков,средняя MAE,разброс по фолдам
9,4,2,0.05,0.18507,0.00439
12,4,3,0.05,0.18514,0.00421
18,8,2,0.05,0.18519,0.00433
16,4,5,0.10,0.18539,0.00445
13,4,3,0.10,0.18557,0.00425
10,4,2,0.10,0.18561,0.00435
15,4,5,0.05,0.18572,0.00437
20,8,2,0.20,0.18574,0.00422
23,8,3,0.20,0.18579,0.00463
19,8,2,0.10,0.18579,0.00444


Выбранные параметры: {'classifier__C': 4, 'features__min_df': 2, 'features__numeric_weight': 0.05}


Выбраны **C=4, min_df=2, вес числовых признаков 0,05**, средняя MAE **0,18507**. Это лучший вариант среди проверенных сочетаний.

Признаки и параметры выбирались по этим данным. Поэтому это результат кросс-валидации при подборе модели; отдельного независимого теста или результата Kaggle нет.

## 7. Сохранение для API

При `refit=True` поиск сам обучает выбранный вариант на всех 59 976 отзывах. Отдельно обучаем словный baseline для сравнения и переключения через `/load_model`.

`save_models` сохраняет две модели вместе с их словарями и масштабированием. `manifest.json` — короткий каталог для сервиса: имена файлов, версии и контрольные суммы.

In [30]:
final_model = search.best_estimator_

word_model_for_api = build_pipeline(word_only=True)
word_model_for_api.fit(reviews, labels)

save_models(final_model, word_model_for_api, root / "models")

In [31]:
saved_model = joblib.load(root / "models/tfidf.joblib")
examples = pd.Series(
    [
        "Excellent service and fast delivery!",
        "My order never arrived and nobody replied.",
    ]
)

display(
    pd.DataFrame(
        {
            "отзыв": examples,
            "прогноз": saved_model.predict(examples),
        }
    )
)

,отзыв,прогноз
0,Excellent service and fast delivery!,5
1,My order never arrived and nobody replied.,1


## Итог

- Постоянная медиана: MAE 1,43781; словный baseline: 0,19008; итоговая модель: **0,18507** на пяти фолдах.
- Итоговая модель — TF-IDF по словам и символам + 13 числовых признаков + Logistic Regression. Стоп-слова сохраняются.
- Итоговые `C=4`, `min_df=2` и вес числовых признаков 0,05 выбраны совместно через `GridSearchCV`. Ограничения словарей и длины символьных фрагментов зафиксированы как практические настройки.
- Основная оставшаяся слабая сторона — различение оценок 2–4.